In [1]:
!pip install pypdf sentence-transformers faiss-cpu groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.0 MB/s eta 0:00:00


In [2]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [3]:
from google.colab import files

uploaded = files.upload()

Saving ACM_Vector_RAG.pdf to ACM_Vector_RAG.pdf


In [6]:
reader = PdfReader("ACM_Vector_RAG.pdf")
len(reader.pages)
print("Number of pages:", len(reader.pages))

Number of pages: 21


In [7]:
text = ""

for page in reader.pages:
    text += page.extract_text()

In [8]:
print(text[:2000])

ApexBank Retail Banking Guide
Customer Reference Manual for Individual Banking Services
Edition 4.2
Importantnotice: thisdocumentisentirelyfictional.ApexBankisamade-upinstitution
created for training and demonstration purposes. Every policy, product, interest rate, fee, limit,
time frame, telephone number, e-mail address, website, procedure, and name mentioned in this
guide is invented. None of it describes a real bank, a real regulator, or real legal rights, and
nothing in it should be relied upon for real financial decisions. All amounts are expressed in
Apex Dollars, written as AXD, which is also a fictional currency. Where the guide mentions
taxes, regulators, or schemes, these are likewise fictional stand-ins.
1 Introduction to ApexBank
ApexBank was founded in 1987 in Meridian City and has grown into a mid-sized retail bank
serving about 6.8 million customers. It operates 412 branches, 2,850 ATMs, and a network of
cash deposit machines, and it offers digital banking through its mo

In [9]:
chunk_size = 500

chunks = []

for i in range(0, len(text), chunk_size):
    chunk = text[i:i + chunk_size]
    chunks.append(chunk)

In [11]:
print("Number of chunks:", len(chunks))
print(chunks[0])
print("ended here")
print(chunks[1])

Number of chunks: 137
ApexBank Retail Banking Guide
Customer Reference Manual for Individual Banking Services
Edition 4.2
Importantnotice: thisdocumentisentirelyfictional.ApexBankisamade-upinstitution
created for training and demonstration purposes. Every policy, product, interest rate, fee, limit,
time frame, telephone number, e-mail address, website, procedure, and name mentioned in this
guide is invented. None of it describes a real bank, a real regulator, or real legal rights, and
nothing in it should be relied u
ended here
pon for real financial decisions. All amounts are expressed in
Apex Dollars, written as AXD, which is also a fictional currency. Where the guide mentions
taxes, regulators, or schemes, these are likewise fictional stand-ins.
1 Introduction to ApexBank
ApexBank was founded in 1987 in Meridian City and has grown into a mid-sized retail bank
serving about 6.8 million customers. It operates 412 branches, 2,850 ATMs, and a network of
cash deposit machines, and it off

In [12]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)
embeddings = embedding_model.encode(chunks)
print(embeddings.shape)
print(embeddings[0])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(137, 384)
[-8.42069276e-03 -3.75186726e-02 -1.67358726e-01 -5.38585000e-02
 -5.36469035e-02 -8.26095045e-03  3.33118588e-02  1.16616189e-02
 -4.60508354e-02  7.74804456e-03 -1.83431767e-02  5.07832132e-02
 -6.06891587e-02 -5.05924486e-02  2.49306541e-02 -3.23241577e-02
  5.66082401e-03 -3.08518279e-02  2.73742210e-02  1.42782852e-02
  8.22265148e-02 -1.17851682e-02 -6.44002184e-02  2.16402784e-02
 -4.54748832e-02 -8.82146508e-02 -6.61819521e-03 -5.23225106e-02
 -2.34985556e-02 -5.69744259e-02 -4.78275083e-02  3.93478088e-02
  4.47908230e-02 -1.09437965e-01  6.32008612e-02 -4.77414429e-02
  7.79355913e-02  3.65761407e-02  1.13542844e-02  3.36915115e-03
 -5.36401682e-02 -3.56403328e-02 -6.02832921e-02 -2.89155021e-02
  6.92186579e-02  7.29791597e-02 -1.20846201e-02  4.85129841e-02
 -3.51672880e-02  5.29271103e-02 -3.26673985e-02 -5.99382147e-02
  2.16569453e-02 -3.68716791e-02  1.19928243e-02  2.20788401e-02
 -5.34878150e-02 -2.80453125e-03  4.71374243e-02 -4.16462570e-02
  5.92952780e-

In [13]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

In [14]:
index.add(
    np.array(embeddings).astype("float32")
)

In [15]:
question = "What should I do if someone asks for my OTP?"

In [16]:
question_embedding = embedding_model.encode(
    [question]
)

In [17]:
distances, indices = index.search(
    np.array(question_embedding).astype("float32"),
    3
)

In [18]:
print(indices)
print(distances)
for i in indices[0]:
    print("-----")
    print(chunks[i])

[[ 74  75 132]]
[[0.9284414 0.9701118 0.9741787]]
-----
 not received
an OTP, the cause is most often an incorrect registered number, poor network coverage, a full
message inbox, or a do-not-disturb setting that blocks service messages.
No employee of ApexBank, and no one acting for the bank, will ever ask for an OTP, a card
PIN, the CVV on the back of the card, an MPIN, or a password. This applies to telephone calls,
SMS, e-mail, instant messages, and visits. Anyone who asks for this information is a fraudster,
even if they know the customer’s name,
-----
 account number, or recent transactions, and even if the
caller identification seems to show the bank’s number. A customer who is asked for an OTP
must not give it, should end the conversation, and should report the attempt immediately to
the fraud line at 1800-555-0199 or through the Report Fraud option in the app. If an OTP has
been shared already, the customer should freeze the account in the app or call the hotline atonce, as eve

In [19]:
relevant_chunks = []

for i in indices[0]:
    relevant_chunks.append(chunks[i])

context = "\n\n".join(relevant_chunks)


In [20]:
prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}
"""

In [22]:
from google.colab import userdata

api_key =("yaad se daaldena")

In [23]:
from groq import Groq

client = Groq(api_key=api_key)

In [33]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

In [34]:
answer = response.choices[0].message.content

print(answer)

If someone asks you for your OTP, do **not** give it to them.  
- End the conversation immediately.  
- Report the attempt right away to ApexBank’s fraud line at **1800‑555‑0199** or use the “Report Fraud” option in the app.  
- If you have already shared the OTP, freeze your account immediately (via the app or by calling the hotline).


In [36]:
def ask_rag(question):

    question_embedding = embedding_model.encode([question])

    distances, indices = index.search(
        np.array(question_embedding).astype("float32"),
        3
    )

    relevant_chunks = []

    for i in indices[0]:
        relevant_chunks.append(chunks[i])

    context = "\n\n".join(relevant_chunks)

    prompt = f"""
    Answer the question using only the context below.

    Context:
    {context}

    Question:
    {question}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [37]:
import gradio as gr

demo = gr.Interface(
    fn=ask_rag,
    inputs=gr.Textbox(
        label="Ask a question",
        placeholder="Ask something about the document..."
    ),
    outputs=gr.Textbox(
        label="RAG Assistant"
    ),
    title="ACM RAG Assistant",
    description="Ask questions about the uploaded document."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5eed2e7997fd77cc0c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
